In [9]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [10]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber/"

In [11]:
total_df =  pd.read_csv("{}schreiber_total_data.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(total_df['allele']))

per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_mt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    per_allele_wt_df[allele_name] = pd.read_csv("{}schreiber_data_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";")
    
def defineOrganism(allele_name):
    for organism_to_use in PER_ORGANISM_PER_CLASS_ALLELES.keys():
        per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use]
        for class_to_use, alleles in per_class_alleles.items():
            if allele_name in alleles:
                return organism_to_use
    print("organism not  found")
    return None


def defineClass(allele_name):
    organism =  defineOrganism(allele_name)
    per_class_alleles = PER_ORGANISM_PER_CLASS_ALLELES[organism]
    for class_to_use, alleles in per_class_alleles.items():
        if allele_name in alleles:
            return class_to_use
    print("not found")
    return None

    
ALLELES

['H-2Ld', 'H-2IEd']

# 1. Read all the predictions

In [12]:
per_method_columns = {
    'NetMHC': ['netMHC-3.4_score', 'netMHC-3.4_affinity', 'netMHC-4.0_score',  'netMHC-4.0_affinity','netMHC-4.0_percentile_rank', \
               'netMHCpan-4.1_score_ba', 'netMHCpan-4.1_score_el', 'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 'netMHCpan-4.1_percentile_rank_el'  ],
    'netMHC-3.4': [ 'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':[ 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':[ 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcnuggets':[ 'score', 'affinity', 'percentile_rank'],
    'acme':['score'],
    'puffin':['score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood'],
    'mhcseqnet':['score'],
    'mhcattnnet':[ 'score']
}
methods = ['NetMHC', 'puffin', 'mhcnuggets', 
           #'acme', 'mhcseqnet', 'mhcattnnet'
          ]

In [13]:
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber/"

per_allele_per_method_predictions = dict()

less_is_good_columns = {'affinity','percentile_rank', 'netMHC-3.4_affinity', 'netMHC-4.0_affinity', 
                                    'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 
                                    'netMHCpan-4.1_percentile_rank_el', 'netMHC-4.0_percentile_rank', 'epistemic_var', 'aleatoric_var' }


for allele_name in ALLELES:
    print(f"{allele_name}:", end=" ")
    per_allele_per_method_predictions[allele_name] = dict()
    for method in methods:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        prediction_df = pd.read_csv(f"{PREDICTION_PATH}{method}/{method}_predictions_allele_{target_allele_name}.csv", sep=';')

        #add diff 

        print(f"{method}", end=" ")
        for score_column in per_method_columns[method]:
            result_score_column = f"{method}_{score_column}" if not (method == 'NetMHC') else score_column
            wt_scores = prediction_df[result_score_column + '_wt'].values
            mt_scores = prediction_df[result_score_column + '_mt'].values
            wt_peptide =  prediction_df['peptide_wt'].values
            # for these columns mt shoud be less than wt. In this case wt/mt will be > 1 and this is good. 
            if score_column in less_is_good_columns:
                advantage = np.array([wt_sc/(mt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            # for these columns mt shoud be greater than wt. In this case mt/wt will be > 1 and this is good. 
            else:
                advantage = np.array([mt_sc/(wt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            prediction_df[f"{result_score_column}_advantage"] = advantage



        per_allele_per_method_predictions[allele_name][method] = prediction_df
    print()
            

H-2Ld: NetMHC puffin mhcnuggets 
H-2IEd: NetMHC puffin mhcnuggets 


# 2. Merge predictions

In [15]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    list_for_allele = list(per_allele_per_method_predictions[allele_name].values())

    result_df = list_for_allele[0].copy()
    for df_to_merge in list_for_allele[1:]:
        result_df = result_df.merge(df_to_merge, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'], how='left')
    result_df = result_df[['allele'] + [column for column in result_df.columns if column.endswith('mt')] 
                          + [column for column in result_df.columns if column.endswith('_advantage')] 
                          + [column for column in result_df.columns if column.endswith('wt')]] 
    per_allele_predictions[allele_name] = result_df


In [17]:
per_allele_per_method_predictions[allele_name][methods[1]]

,peptide_mt,length,allele,source_sequence_name,puffin_score_mt,puffin_epistemic_var_mt,puffin_aleatoric_var_mt,puffin_binding_likelihood_mt,peptide_wt,puffin_score_wt,puffin_epistemic_var_wt,puffin_aleatoric_var_wt,puffin_binding_likelihood_wt,puffin_score_advantage,puffin_epistemic_var_advantage,puffin_aleatoric_var_advantage,puffin_binding_likelihood_advantage
0,NTGRGYVLI,9,H-2IEd,seq0,0.016322,0.000026,0.001287,0.000000e+00,NTGRGYVLI,0.016322,0.000026,0.001287,0.000000e+00,1.0,0.999996,1.0,0.000000
1,TGRGYVLIR,9,H-2IEd,seq1,0.016898,0.000029,0.001409,0.000000e+00,TGRGYVLIR,0.016898,0.000029,0.001409,0.000000e+00,1.0,0.999997,1.0,0.000000
2,GRGYVLIRI,9,H-2IEd,seq2,0.031087,0.000448,0.003077,5.716538e-13,GRGYVLIRI,0.031087,0.000448,0.003077,5.716538e-13,1.0,1.000000,1.0,0.005684
3,RGYVLIRIL,9,H-2IEd,seq3,0.041623,0.000739,0.004703,1.076685e-08,RGYVLIRIL,0.041623,0.000739,0.004703,1.076685e-08,1.0,1.000000,1.0,0.990798
4,GYVLIRILR,9,H-2IEd,seq4,0.049966,0.001038,0.005438,1.752516e-07,GYVLIRILR,0.049966,0.001038,0.005438,1.752516e-07,1.0,1.000000,1.0,0.999430
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,RPDNTGRGYVLIRILRRAVRYSH,23,H-2IEd,seq129,0.409938,0.015389,0.052810,4.727871e-01,RPDNTGRGYVLIRILRRAVRYSH,0.409938,0.015389,0.052810,4.727871e-01,1.0,1.000000,1.0,1.000000
130,PDNTGRGYVLIRILRRAVRYSHE,23,H-2IEd,seq130,0.425708,0.015808,0.052997,5.001439e-01,PDNTGRGYVLIRILRRAVRYSHE,0.425708,0.015808,0.052997,5.001439e-01,1.0,1.000000,1.0,1.000000
131,GRPDNTGRGYVLIRILRRAVRYSH,24,H-2IEd,seq131,0.382432,0.015585,0.053169,4.257051e-01,GRPDNTGRGYVLIRILRRAVRYSH,0.382432,0.015585,0.053169,4.257051e-01,1.0,1.000000,1.0,1.000000
132,RPDNTGRGYVLIRILRRAVRYSHE,24,H-2IEd,seq132,0.409256,0.015362,0.054805,4.721271e-01,RPDNTGRGYVLIRILRRAVRYSHE,0.409256,0.015362,0.054805,4.721271e-01,1.0,1.000000,1.0,1.000000


## Save total pred to file

In [18]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)